# 01. Exploratory Data Analysis (EDA)
## Customer Sentiment Analysis & Reporting for E-commerce

This notebook provides an in-depth exploration of the E-commerce Clothing Reviews dataset.
Key objectives:
- Validate dataset schema and missing value distributions
- Derive 3-class sentiment labels (negative, neutral, positive) from customer ratings
- Analyze text lengths and token distributions across sentiment classes
- Inspect departmental differences in customer satisfaction
- Identify high-frequency terms and complaint themes

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

from src.config import FIGURES_DIR
from src.data.load import load_raw_data, validate_schema, get_data_summary
from src.data.clean import clean_data, generate_data_quality_report
from src.data.eda import generate_eda_figures

sns.set_theme(style='whitegrid')

### 1. Data Ingestion & Schema Validation

In [ ]:
raw_df = load_raw_data()
print(f'Raw dataset shape: {raw_df.shape}')
is_valid, validation_errors = validate_schema(raw_df)
print(f'Schema valid: {is_valid}')
if validation_errors:
    print('Validation notes:', validation_errors)
print('Missing values per column:', get_data_summary(raw_df)['missing_values'])

raw_df.head(3)

### 2. Data Cleaning & Sentiment Label Derivation
Ratings mapping:
- **1 or 2 stars** -> `negative`
- **3 stars** -> `neutral`
- **4 or 5 stars** -> `positive`
- Rows missing Review Text are dropped; whitespace is normalized.

In [ ]:
clean_df, metrics = clean_data(raw_df)
generate_data_quality_report(metrics)
print(f'Clean dataset shape: {clean_df.shape}')
print(f'Dropped empty reviews: {metrics["dropped_empty_text"]}')
print(f'Dropped duplicates: {metrics["dropped_duplicates"]}')
print(f'Label counts:\n{clean_df["true_label"].value_counts()}')

### 2b. NumPy: Vectorised Statistics
Review lengths and ratings as NumPy arrays: summary statistics and correlation without Python loops.

In [ ]:
lengths = clean_df['Review Text'].str.len().to_numpy()
ratings = clean_df['Rating'].to_numpy()

print(f'Length mean={lengths.mean():.1f}, median={np.median(lengths):.0f}, p95={np.percentile(lengths, 95):.0f}')
print(f'Pearson r(length, rating) = {np.corrcoef(lengths, ratings)[0, 1]:.3f}')
for r in np.unique(ratings):
    print(f'  rating {r}: mean length {lengths[ratings == r].mean():.1f} chars')

### 3. Visualizing Key Distributions
We generate standardized figures saved directly to `reports/figures/`.

In [ ]:
generate_eda_figures(clean_df)
for name in ['label_distribution', 'text_length_distribution', 'rating_by_department', 'top_terms_per_class']:
    display(Image(filename=str(FIGURES_DIR / f'{name}.png'), width=700))

### 4. Summary Findings
- **Class Imbalance**: Positive reviews dominate (77.1%), neutral 12.5%, negative 10.5% (see `reports/data_quality.md`). Macro-F1 and `class_weight="balanced"` are used because of this.
- **Department Differences**: compare median ratings in `rating_by_department.png`; departments with wider low-rating tails deserve drill-down in the dashboard.
- **Text Lengths**: compare the per-rating mean lengths above and `text_length_distribution.png` before drawing conclusions about verbosity.